# Reporte final: clasificación de 104 tipos de flores

Este notebook reproduce **de principio a fin** la tabla de resultados del reporte final
(`reports/final/resultados.csv`) y todas sus tablas y figuras.

- Repositorio: <https://github.com/Lunaaaalj/NeuralNetworksTeamEnvironment>
- Datos: competencia de Kaggle [Petals to the Metal](https://www.kaggle.com/competitions/tpu-getting-started) (TFRecords 224×224).
- Métrica principal: **F1 macro** sobre las 3,712 imágenes de validación (la misma métrica de Kaggle).

**Cómo correrlo**

```bash
uv venv --python 3.12 .venv && source .venv/bin/activate
uv pip install -r requirements.txt
python scripts/download_data.py          # requiere token de Kaggle
jupyter nbconvert --to notebook --execute --inplace notebooks/reporte_final.ipynb
```

Con `RUN_TRAINING = False` (valor por defecto) el notebook **no entrena**: reconstruye la tabla
a partir de los `history.csv`, `summary.json` y `eval_report.json` versionados en
`reports/final/experimentos/runs/`. Con `RUN_TRAINING = True` vuelve a entrenar y evaluar los
7 experimentos del reporte (≈2.5 h en una Mac M4 con `tensorflow-metal`; las corridas ya terminadas se conservan).


In [ ]:
import json, subprocess, sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # ejecutado desde notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

RUN_TRAINING = False   # True: reentrena los 7 experimentos del reporte (~2.5 h)
EXPERIMENTS = ROOT / "reports/final/experimentos"
REPORT = ROOT / "reports/final"
TFRECORDS = ROOT / "data/raw/tpu-getting-started/tfrecords-jpeg-224x224"
print("Raíz del repositorio:", ROOT)
print("TFRecords disponibles:", TFRECORDS.exists())

## 1. Datos y desbalance de clases

Se cuentan las imágenes por clase leyendo solo la etiqueta de cada TFRecord. Si los datos no
están descargados, se usa el conteo versionado en `reports/final/experimentos/conteo_por_clase.csv`.

In [ ]:
if TFRECORDS.exists():
    from src.classes import CLASSES
    from src.data import class_counts
    counts = pd.DataFrame({"label": range(104), "name": CLASSES,
                           "train": class_counts(TFRECORDS, "train"), "val": class_counts(TFRECORDS, "val")})
    counts.to_csv(EXPERIMENTS / "conteo_por_clase.csv", index=False)
else:
    counts = pd.read_csv(EXPERIMENTS / "conteo_por_clase.csv")
display(counts[["train", "val"]].describe().loc[["count", "min", "50%", "mean", "max"]].round(1))
display(counts.sort_values("train").iloc[[0, 1, 2, -3, -2, -1]])
print(f"Razón máx/mín en entrenamiento: {counts.train.max() / counts.train.min():.1f}x")

## 2. Experimentos

Cada experimento cambia **una sola cosa** respecto a la CNN de referencia
(`configs/final.yaml`: 4 convoluciones, dropout 0.5, aumento de datos, AdamW 0.01, LR 1e-3 con
reducción en meseta, 112×112, semilla 0). Los YAML derivados quedan en
`reports/final/experimentos/configs/`.

In [ ]:
from scripts.run_final import EXPERIMENTS as MATRIX
display(pd.DataFrame([(label, group, description) for label, group, description, _ in MATRIX],
                     columns=["experimento", "grupo", "cambio respecto a la referencia"]))

In [ ]:
if RUN_TRAINING:
    subprocess.run([sys.executable, "-u", "-m", "scripts.run_final"], cwd=ROOT, check=True)
else:
    done = sorted(p.parent.parent.name for p in (EXPERIMENTS / "runs").glob("*/*/eval_report.json"))
    print(f"Se usan {len(done)} corridas guardadas:", ", ".join(done))

## 3. Tabla de resultados

`scripts/analyze_final.py` lee las corridas y escribe `resultados.csv`, las tablas LaTeX
(`reports/final/tablas/`) y las figuras (`reports/final/figures/`). Todas las métricas se
calculan en el checkpoint con mejor F1 macro de validación. *F1 entr.* se mide sobre 3,712 imágenes
fijas de entrenamiento **sin** aumento ni dropout, en esa misma época; la brecha es F1 entr. − F1 val.

In [ ]:
subprocess.run([sys.executable, "-m", "scripts.analyze_final"], cwd=ROOT, check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
results = pd.read_csv(REPORT / "resultados.csv")
columns = ["experimento", "grupo", "parametros", "mejor_epoca", "epocas", "val_exactitud", "val_f1_macro",
           "train_f1_macro", "brecha_f1", "segundos_entrenamiento"]
display(results[columns].sort_values("val_f1_macro", ascending=False).reset_index(drop=True))

In [ ]:
display(Image(filename=str(REPORT / "figures/comparacion_experimentos.png"), width=800))

## 4. Curvas de entrenamiento y validación (B1)

Pérdida, exactitud y F1 macro por época. La línea punteada gris son las métricas que reporta
`fit()` (con aumento y dropout activos); la azul, el mismo subconjunto de entrenamiento evaluado
sin ellos, que sí es comparable con validación (naranja). La línea vertical marca el checkpoint.

In [ ]:
display(Image(filename=str(REPORT / "figures/curvas_modelos.png"), width=900))

## 5. Métricas por clase y desbalance (B2, B3)

In [ ]:
display(Image(filename=str(REPORT / "figures/distribucion_clases.png"), width=800))
reference = json.loads(next((EXPERIMENTS / "runs/referencia").glob("*/eval_report.json")).read_text())
per_class = pd.DataFrame(reference["per_class"]).merge(counts[["label", "train"]], on="label")
ordered = per_class.sort_values(["f1", "train"], ascending=[False, False])
display(Markdown("**CNN de referencia: 10 mejores clases**")); display(ordered.head(10).reset_index(drop=True))
display(Markdown("**CNN de referencia: 10 peores clases**")); display(ordered.tail(10).iloc[::-1].reset_index(drop=True))
print("Correlación de Spearman entre tamaño de clase y F1:",
      round(per_class[["train", "f1"]].corr(method="spearman").iloc[0, 1], 3))

In [ ]:
display(Image(filename=str(REPORT / "figures/f1_vs_tamano_clase.png"), width=800))
display(results.set_index("experimento").loc[["referencia", "class_weight"],
        ["val_exactitud", "val_f1_macro", "f1_10_clases_menores", "f1_10_clases_mayores", "clases_f1_cero"]])

## 6. Regularización con evidencia (B4) e hiperparámetros (B5)

In [ ]:
display(Image(filename=str(REPORT / "figures/curvas_regularizacion.png"), width=900))
regularization = ["referencia", "sin_dropout", "sin_aumento", "sin_weight_decay", "sin_regularizacion", "l2", "label_smoothing"]
hyper = ["referencia", "lr_3e-4", "dropout_0.3", "res_160"]
table = results.set_index("experimento")
display(Markdown("**Ablación de regularización**"))
display(table.loc[[k for k in regularization if k in table.index],
                  ["dropout", "aumento", "optimizador", "l2", "label_smoothing", "val_f1_macro", "train_f1_macro", "brecha_f1"]])
display(Markdown("**Hiperparámetros (uno a la vez)**"))
display(table.loc[[k for k in hyper if k in table.index],
                  ["lr", "dropout", "img_size", "val_exactitud", "val_f1_macro", "brecha_f1", "segundos_entrenamiento"]])

## 7. MLP vs. CNN vs. transfer learning (B6, B7)

In [ ]:
display(table.loc[[k for k in ["mlp", "referencia", "cnn_ampliada"] if k in table.index],
                  ["parametros", "img_size", "val_exactitud", "val_f1_macro", "train_f1_macro", "brecha_f1"]])